# **Abstracción**
Se presenta una serie de ejemplos ordenados de menor a mayor complejidad, que demuestran cómo la abstracción permite definir contratos claros, reutilizar código y diseñar sistemas modulares y mantenibles.

### **Ejemplo 1:** Abstracción Básica (El plano geométrico)

Concepto: Introducción a la clase abstracta utilizando el módulo integrado abc. Define un contrato simple donde se impide la instanciación directa de la clase base y se obliga a las subclases a resolver el "cómo" matemático.

**Abstracción:** El resto de la aplicación interactúa con objetos de tipo **Figura** invocando .calcular_area(), ignorando por completo si se trata de un círculo, un cuadrado o cualquier otra forma compleja.

In [ ]:
from abc import ABC, abstractmethod
import math

class Figura(ABC):
    """Clase abstracta que actúa como molde para cualquier figura geométrica."""

    @abstractmethod
    def calcular_area(self) -> float:
        """Método abstracto. Las subclases deben definir cómo calcular su área."""
        pass

    @abstractmethod
    def perimetro(self) -> float:
        """Método que calcula el perímetro de la figura."""
        pass

class Circulo(Figura):
    def __init__(self, radio: float):
        self.radio = radio

    def calcular_area(self) -> float:
        # Implementación de la fórmula específica para el círculo
        return math.pi * (self.radio ** 2)

    def perimetro(self) -> float:
        # Implementación de la fórmula específica para el círculo
        return 2 * math.pi * self.radio

class Rectangulo(Figura):
    def __init__(self, base: float, altura: float):
        self.base = base
        self.altura = altura

    def calcular_area(self) -> float:
        # Implementación de la fórmula específica para el rectángulo
        return self.base * self.altura

    def perimetro(self) -> float:
        # Implementación de la fórmula específica para el rectángulo
        return 2 * (self.base + self.altura)

# triangulo

# Uso del código:
# figura = Figura()  # Esto lanzará un TypeError automáticamente en Python.
circulo = Circulo(5.0)
print(f"Área del círculo: {circulo.calcular_area():.2f}")
print(f"Perimetro es {circulo.perimetro}")

rectangulo = Rectangulo(4.0, 6.0)
print(f"Área del rectángulo: {rectangulo.calcular_area():.2f}")
print(f"Perimetro es {rectangulo.perimetro}")

Área del círculo: 78.54
Perimetro es <bound method Circulo.perimetro of <__main__.Circulo object at 0x7e35bd8aa900>>
Área del rectángulo: 24.00
Perimetro es <bound method Rectangulo.perimetro of <__main__.Rectangulo object at 0x7e35bd8aaba0>>


### **Ejemplo 2:** Abstracción con Estado Común e Inicialización (Gestión de Empleados)

Concepto: Uso de un constructor "_ _init_ _" en la clase abstracta para compartir atributos comunes de datos y el uso de super() en las subclases para evitar la duplicación de código de inicialización.

**Abstracción:** El sistema de nómina no necesita saber qué tipo de contrato legal tiene cada empleado para emitir sus pagos; la clase abstracta unifica la interfaz.

In [ ]:
from abc import ABC, abstractmethod

class Empleado(ABC):
    def __init__(self, nombre: str, salario_base: float, bonos: int):
        self.nombre = nombre
        self.salario_base = salario_base

    @abstractmethod
    def calcular_pago_neto(self) -> float:
        """Contrato para calcular el sueldo neto tras deducciones o bonos."""
        pass

class EmpleadoPlanta(Empleado):
    def __init__(self, nombre: str, salario_base: float, bono_antiguedad: float, bonos: int):
        # Inicializa los atributos comunes usando la clase base
        super().__init__(nombre, salario_base, bonos)
        self.bono_antiguedad = bono_antiguedad


    def calcular_pago_neto(self) -> float:
        # Lógica de cálculo específica
        return self.salario_base + self.bono_antiguedad

class EmpleadoFreelance(Empleado):
    def __init__(self, nombre: str, salario_base: float, retencion_impuestos: float, bonos: int):
        super().__init__(nombre, salario_base, bonos)
        self.retencion_impuestos = retencion_impuestos

    def calcular_pago_neto(self) -> float:
        # Lógica de cálculo específica para contratistas externos
        return self.salario_base * (1 - self.retencion_impuestos)

# Uso del código:
nomina = [
    EmpleadoPlanta("Carlos", 2500.0, 300.0, 5),
    EmpleadoFreelance("Sofía", 1800.0, 0.15, 8)
]

for emp in nomina:
    print(f"Empleado: {emp.nombre} | Pago Neto: ${emp.calcular_pago_neto():.2f}")


Empleado: Carlos | Pago Neto: $2800.00
Empleado: Sofía | Pago Neto: $1530.00


### **Ejemplo 3:** Propiedades Abstractas (Sistemas de Plugins Multimedia)

**Concepto:** Combinación del decorador @property con @abstractmethod. Esto permite exigir a las subclases que no solo implementen métodos, sino que expongan de manera obligatoria ciertas variables o atributos de solo lectura.

**Por qué es Abstracción:** Permite construir un reproductor de música modular que cargue dinámicamente archivos externos y pueda validar sus metadatos (codec_name) antes de llamar al decodificador.

In [ ]:
from abc import ABC, abstractmethod

class PluginAudio(ABC):

    @property
    @abstractmethod
    def codec_name(self) -> str:
        """Debe retornar el nombre del codec de audio."""
        pass

    @abstractmethod
    def decodificar(self, archivo: str) -> bytes:
        """Descomprime el archivo de audio en bytes crudos."""
        pass

class ReproductorMP3(PluginAudio):
    @property
    def codec_name(self) -> str:
        return "MPEG-1 Audio Layer III"

    def decodificar(self, archivo: str) -> bytes:
        return f"Procesando algoritmo MP3 para {archivo}...".encode('utf-8')

# Uso del código:
reproductor = ReproductorMP3()
print(f"Codec activo: {reproductor.codec_name}")

Codec activo: MPEG-1 Audio Layer III


### **Ejemplo 4:** Abstracción de Capas de Datos (Persistencia Intercambiable)**

**Concepto:** Ocultación de los detalles de infraestructura técnica. Los estudiantes aprenden cómo desacoplar la lógica de la aplicación del motor de almacenamiento real (por ejemplo, base de datos local vs. nube).

**Por qué es Abstracción:** Si el día de mañana decides migrar los datos a MongoDB o PostgreSQL, solo tendrás que escribir una nueva subclase que cumpla con `RepositorioUsuarios` sin tocar una sola línea de código de tu aplicación principal.

In [ ]:
from abc import ABC, abstractmethod
from typing import Dict, Any

class RepositorioUsuarios(ABC):
    @abstractmethod
    def guardar_usuario(self, user_id: str, datos: Dict[str, Any]) -> None:
        pass

    @abstractmethod
    def obtener_usuario(self, user_id: str) -> Dict[str, Any]:
        pass

class RepositorioMemoria(RepositorioUsuarios):
    """Implementación rápida para entornos de prueba (Test/Mock)."""
    def __init__(self):
        self._db = {}

    def guardar_usuario(self, user_id: str, datos: Dict[str, Any]) -> None:
        self._db[user_id] = datos

    def obtener_usuario(self, user_id: str) -> Dict[str, Any]:
        return self._db.get(user_id, {})

# Uso del código:
# Un controlador de la app interactúa únicamente con el 'RepositorioUsuarios' abstracto.
repo: RepositorioUsuarios = RepositorioMemoria()
repo.guardar_usuario("usr_01", {"nombre": "Ana", "rol": "Admin"})
print(repo.obtener_usuario("usr_01"))

{'nombre': 'Ana', 'rol': 'Admin'}


### **Ejemplo 5:** El Patrón "Template Method" (Flujo de Procesamiento Estructurado)

**Concepto:** La clase abstracta define el "esqueleto" o algoritmo maestro de un proceso mediante un método concreto, pero delega los pasos específicos de ejecución a sus subclases abstractas.

**Por qué es Abstracción:** La estructura del flujo de procesamiento está garantizada y protegida en la clase base, lo que impide que las subclases alteren el orden de los pasos lógicos.

In [ ]:
from abc import ABC, abstractmethod

class ProcesadorReportes(ABC):
    def generar_reporte(self, archivo_origen: str) -> None:
        """Método plantilla que define la secuencia exacta de pasos."""
        print("--- Iniciando ciclo de reporte ---")
        datos = self._leer_origen(archivo_origen)
        datos_limpios = self._limpiar_datos(datos)
        self._exportar(datos_limpios)
        print("--- Reporte finalizado con éxito ---")

    @abstractmethod
    def _leer_origen(self, archivo: str) -> str:
        pass

    @abstractmethod
    def _limpiar_datos(self, datos_crudos: str) -> str:
        pass

    @abstractmethod
    def _exportar(self, datos_procesados: str) -> None:
        pass

class ReporteHTML(ProcesadorReportes):
    def _leer_origen(self, archivo: str) -> str:
        return f"[Datos crudos de {archivo}]"

    def _limpiar_datos(self, datos_crudos: str) -> str:
        return datos_crudos.replace("[", "").replace("]", "")

    def _exportar(self, datos_procesados: str) -> None:
        print(f"<html><body>{datos_procesados}</body></html>")

# Uso del código:
generador = ReporteHTML()
generador.generar_reporte("ventas.csv")

--- Iniciando ciclo de reporte ---
<html><body>Datos crudos de ventas.csv</body></html>
--- Reporte finalizado con éxito ---


### **Ejemplo 6:** Pipelines de Datos ETL Complejos (Abstracción a Nivel Profesional)

**Concepto:** Un flujo ETL (*Extract, Transform, Load*) completo que procesa datos reales por lotes, aplicando el manejo de excepciones, logs y validación de tipos en entornos de producción.

**Por qué es Abstracción:** El orquestador general de flujos de trabajo (*scheduler*) solo necesita invocar al método público `.ejecutar()`. No tiene interés en saber si los datos provienen de un archivo JSON, una API web o cómo se transforman internamente.


In [ ]:
from abc import ABC, abstractmethod
from typing import List, Dict

class PipelineETL(ABC):
    """Abstracción empresarial para pipelines de ciencia de datos."""

    def __init__(self, nombre_pipeline: str):
        self.nombre_pipeline = nombre_pipeline

    @abstractmethod
    def extraer(self) -> List[Dict[str, str]]:
        """Extrae la información desde el origen de datos."""
        pass

    @abstractmethod
    def transformar(self, datos: List[Dict[str, str]]) -> List[Dict[str, str]]:
        """Aplica las reglas de negocio y limpieza técnica."""
        pass

    @abstractmethod
    def cargar(self, datos_transformados: List[Dict[str, str]]) -> None:
        """Carga el resultado en el destino de almacenamiento final."""
        pass

    def ejecutar(self) -> None:
        """Coordina el ciclo ETL completo con control de errores."""
        print(f"Iniciando Pipeline: {self.nombre_pipeline}")
        try:
            raw_data = self.extraer()
            transformed_data = self.transformar(raw_data)
            self.cargar(transformed_data)
            print(f"Pipeline {self.nombre_pipeline} completado de forma limpia.\n")
        except Exception as e:
            print(f"🚨 Error crítico en el pipeline '{self.nombre_pipeline}': {str(e)}")

# Implementación concreta para Procesamiento de Clientes
class PipelineClientes(PipelineETL):
    def extraer(self) -> List[Dict[str, str]]:
        return [{"id": "1", "nombre": "juan perez"}, {"id": "2", "nombre": "MARIA GOMEZ"}]

    def transformar(self, datos: List[Dict[str, str]]) -> List[Dict[str, str]]:
        # Formatea los nombres de forma homogénea
        for d in datos:
            d["nombre"] = d["nombre"].title()
        return datos

    def cargar(self, datos_transformados: List[Dict[str, str]]) -> None:
        print(f"Guardando {len(datos_transformados)} registros en la base de datos central:")
        for registro in datos_transformados:
            print(f" -> Insertado: {registro}")

# Uso del código:
pipeline_ventas = PipelineClientes("Carga Diaria de Clientes")
pipeline_ventas.ejecutar()

Iniciando Pipeline: Carga Diaria de Clientes
Guardando 2 registros en la base de datos central:
 -> Insertado: {'id': '1', 'nombre': 'Juan Perez'}
 -> Insertado: {'id': '2', 'nombre': 'Maria Gomez'}
Pipeline Carga Diaria de Clientes completado de forma limpia.



# **DESAFÍOS DE ABSTRACCIÓN**

**Recordatorio:** La abstracción es uno de los pilares de la Programación Orientada a Objetos (POO). Consiste en ocultar los detalles complejos de implementación y mostrar únicamente la interfaz esencial que el usuario necesita para interactuar con un objeto.
En Python, la forma estándar de aplicar abstracción es mediante el módulo abc (Abstract Base Classes), utilizando la clase base ABC y el decorador @abstractmethod.

## **Desafío 1: Sistema de Notificaciones**
**Enunciado:**

Una aplicación necesita enviar alertas a los usuarios a través de distintos canales (por ejemplo, Email y SMS). La aplicación principal no debe preocuparse por cómo funciona cada red de comunicación; solo necesita un método unificado llamado **enviar(mensaje)** que funcione para cualquier tipo de notificación.

**¿Dónde está la abstracción?** El bucle final no sabe si está enviando un correo o un SMS; solo sabe que cualquier objeto derivado de `Notificacion` responde al método `.enviar()`.

## **Desafío 2: Procesador de Pagos**

**Enunciado del problema:**

Una tienda en línea acepta cobros mediante **Tarjeta de Crédito** y **PayPal**. Debes diseñar una estructura donde el carrito de compras solo llame a **`procesar_pago(monto)`** sin importar las pasarelas bancarias o APIs internas que requiera cada medio de pago.

**¿Dónde está la abstracción?** Se oculta la validación de tarjetas o la autenticación por token en PayPal tras una llamada común `procesar_pago()`.

## **Desafío 3: Encendido de Vehículos**

**Enunciado del problema:**

Queremos simular el arranque de diferentes medios de transporte (**Auto** y **Moto**). Aunque la mecánica interna para arrancar un motor de auto (inyección electrónica) es muy distinta a la de una moto (pedal o encendido manual), el conductor solo requiere la acción unificada de **`arrancar()`**.